### while building pipelines call columns by index insted of column name. bcz when results from one pipe comes its is numpy array and not dataframe. so if we use column name it will affect the next pipe.\
#### oe = OrdinalEncoder(categories=[['Mild','Strong']])\
#### X_train_cough = oe.fit_transform(X_train[['cough']])      cause issues in future\
#### X_train_cough = oe.fit_transform(X_train.iloc[:, [2]])    good 

when u train model in pipeline, use fit. if u are not trainging model in pipeline use fit_transform

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split


In [4]:
df= pd.read_csv('/Users/kishan/code/genai/ml/ml_learning/datasets/titanic.csv')

In [5]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
341,342,1,1,"Fortune, Miss. Alice Elizabeth",female,24.0,3,2,19950,263.0000,C23 C25 C27,S
519,520,0,3,"Pavlovic, Mr. Stefo",male,32.0,0,0,349242,7.8958,NaN,S
236,237,0,2,"Hold, Mr. Stephen",male,44.0,1,0,26707,26.0000,NaN,S
90,91,0,3,"Christmann, Mr. Emil",male,29.0,0,0,343276,8.0500,NaN,S
734,735,0,2,"Troupiansky, Mr. Moses Aaron",male,23.0,0,0,233639,13.0000,NaN,S


In [6]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [7]:
df.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
291,1,1,female,19.0,1,0,91.0792,C
36,1,3,male,NaN,0,0,7.2292,C
265,0,2,male,36.0,0,0,10.5000,S
616,0,3,male,34.0,1,1,14.4000,S
680,0,3,female,NaN,0,0,8.1375,Q


In [8]:
X= df.drop(columns=['Survived'])
y= df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
X_train.size,X_test.size

(4984, 1253)

In [10]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [11]:
# Applying imputation for age and embarked columns
imputer_age= SimpleImputer(strategy='mean')
imputer_embarked= SimpleImputer(strategy='most_frequent')

X_train_Age= imputer_age.fit_transform(X_train[['Age']])
X_train_Embarked= imputer_embarked.fit_transform(X_train[['Embarked']])

X_test_Age= imputer_age.transform(X_test[['Age']])
X_test_Embarked= imputer_embarked.transform(X_test[['Embarked']])

In [12]:
X_test_Embarked

array([['C'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['Q'],
       ['S'],
       ['Q'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['Q'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['Q'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['Q'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['C'],
       ['S'],
       ['S'],
       ['S'],
       ['Q'],
       ['S'],
       ['S'],
       ['C'],
       ['S'],
       ['Q'],
       ['S'],
       ['S'],
       ['S'],
       ['S'],
       ['Q'],
       ['S'],
       ['S'],
       ['S'],
       ['C'],
       ['C'],
       ['S'],
      

In [13]:
# One hot encoding for categorical columns
ohe_sex= OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked= OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_sex= ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked= ohe_embarked.fit_transform(X_train_Embarked)

X_test_sex= ohe_sex.transform(X_test[['Sex']])
X_test_embarked= ohe_embarked.transform(X_test_Embarked)

In [17]:
X_train_sex

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]], shape=(712, 2))

In [18]:
X_train_transformed= np.concatenate([X_train_sex, X_train_embarked, X_train_Age, X_train[['Pclass','SibSp','Parch','Fare']].to_numpy()], axis=1)
X_test_transformed= np.concatenate([X_test_sex, X_test_embarked, X_test_Age, X_test[['Pclass','SibSp','Parch','Fare']].to_numpy()], axis=1)

In [21]:
X_test_transformed.size

1790

In [22]:
clf= DecisionTreeClassifier(random_state=42)
clf.fit(X_train_transformed, y_train)
y_pred= clf.predict(X_test_transformed)

In [23]:
y_pred

array([0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1])

In [24]:
from sklearn.metrics import accuracy_score
accuracy= accuracy_score(y_test, y_pred)
accuracy

0.7932960893854749

In [27]:
import pickle
pickle.dump(ohe_sex, open('models/one_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('models/one_embarked.pkl', 'wb'))
pickle.dump(clf, open('models/clf.pkl', 'wb'))